# Sistem Pendukung Keputusan Pemilihan Laptop Menggunakan Metode SAW, TOPSIS, dan WASPAS
Studi kasus ini menggunakan dataset riil **Laptop Price** dari Kaggle (disimpan di file lokal `dataset_laptop_lengkap.csv`). 
Data difilter terlebih dahulu (Budget maksimal 1000 Euro & RAM >= 8GB) untuk rekomendasi yang realistis.

### Setup dan Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Pengambilan Data Riil dan Data Cleaning

In [ ]:
# Mengambil dataset riil langsung dari file lokal
file_dataset = 'dataset_laptop_lengkap.csv'
dataset_mentah = pd.read_csv(file_dataset, encoding='latin-1')

dataset_mentah['Price (Euros)'] = dataset_mentah['Price (Euros)'].astype(str).str.replace(',', '.').astype(float)
dataset_mentah['RAM'] = dataset_mentah['RAM'].astype(str).str.replace('GB', '').astype(int)
dataset_mentah['Weight'] = dataset_mentah['Weight'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
storage_mentah = dataset_mentah[' Storage'].astype(str).str.replace('TB', '000GB')
dataset_mentah['Storage (GB)'] = storage_mentah.str.extract(r'(\d+)').astype(float)
dataset_mentah['Screen Size'] = dataset_mentah['Screen Size'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
dataset_mentah['Alternatif'] = dataset_mentah['Manufacturer'] + ' ' + dataset_mentah['Model Name']

# Filter skenario real: Budget maksimal 1000 Euro dan RAM minimal 8GB
dataset_filter = dataset_mentah[(dataset_mentah['Price (Euros)'] <= 1000) & (dataset_mentah['RAM'] >= 8)]

# Ambil 10 sampel teratas dari hasil filter untuk dihitung
dataframe_laptop = dataset_filter[['Alternatif', 'Price (Euros)', 'RAM', 'Storage (GB)', 'Weight', 'Screen Size']].head(10).copy()
dataframe_laptop.columns = ['Alternatif', 'Harga', 'RAM', 'Storage', 'Berat', 'Layar']
display(dataframe_laptop)

### Pemisahan Matriks Keputusan dan Bobot Kriteria

In [ ]:
matriks_keputusan = dataframe_laptop.iloc[:, 1:].values
bobot_kriteria = np.array([0.35, 0.20, 0.20, 0.15, 0.10])
status_kriteria = ['cost', 'benefit', 'benefit', 'cost', 'benefit']
nama_kriteria = ['Harga', 'RAM', 'Storage', 'Berat', 'Layar']
nama_alternatif = dataframe_laptop['Alternatif'].values

### METODE 1: Simple Additive Weighting (SAW)

### Normalisasi Matriks SAW

In [ ]:
print("=== NORMALISASI MATRIKS SAW ===")
matriks_normalisasi_saw = np.zeros_like(matriks_keputusan, dtype=float)
jumlah_kolom = matriks_keputusan.shape[1]
for kolom in range(jumlah_kolom):
    data_kolom = matriks_keputusan[:, kolom]
    if status_kriteria[kolom] == 'benefit':
        nilai_maksimum = np.max(data_kolom)
        print(f"Kriteria {nama_kriteria[kolom]} [Benefit] -> Max = {nilai_maksimum}")
        for baris in range(len(nama_alternatif)):
            matriks_normalisasi_saw[baris, kolom] = data_kolom[baris] / nilai_maksimum
            print(f"  {nama_alternatif[baris]}: {data_kolom[baris]} / {nilai_maksimum} = {matriks_normalisasi_saw[baris, kolom]:.4f}")
    else:
        nilai_minimum = np.min(data_kolom)
        print(f"Kriteria {nama_kriteria[kolom]} [Cost] -> Min = {nilai_minimum}")
        for baris in range(len(nama_alternatif)):
            matriks_normalisasi_saw[baris, kolom] = nilai_minimum / data_kolom[baris]
            print(f"  {nama_alternatif[baris]}: {nilai_minimum} / {data_kolom[baris]} = {matriks_normalisasi_saw[baris, kolom]:.4f}")
    print()

dataframe_normalisasi_saw = pd.DataFrame(np.round(matriks_normalisasi_saw, 4), columns=nama_kriteria)
display(dataframe_normalisasi_saw)

### Perhitungan Preferensi dan Perangkingan SAW

In [ ]:
print("=== PERHITUNGAN NILAI PREFERENSI SAW ===")
nilai_akhir_saw = np.zeros(len(nama_alternatif))
for baris in range(len(nama_alternatif)):
    detail_hitung = []
    for kolom in range(jumlah_kolom):
        detail_hitung.append(f"({bobot_kriteria[kolom]} * {matriks_normalisasi_saw[baris, kolom]:.4f})")
    
    nilai_akhir_saw[baris] = np.sum(matriks_normalisasi_saw[baris] * bobot_kriteria)
    cetak_detail = " + ".join(detail_hitung)
    print(f"{nama_alternatif[baris]}:")
    print(f"  V = {cetak_detail} = {nilai_akhir_saw[baris]:.4f}\n")

hasil_saw = pd.DataFrame({
    'Alternatif': nama_alternatif,
    'Nilai SAW': nilai_akhir_saw
}).sort_values('Nilai SAW', ascending=False).reset_index(drop=True)
hasil_saw.index = hasil_saw.index + 1
display(hasil_saw)

### METODE 2: Technique for Order Preference by Similarity to Ideal Solution (TOPSIS)

### Normalisasi Matriks TOPSIS

In [ ]:
print("=== NORMALISASI MATRIKS TOPSIS ===")
pembagi_topsis = np.zeros(jumlah_kolom)
matriks_normalisasi_topsis = np.zeros_like(matriks_keputusan, dtype=float)

for kolom in range(jumlah_kolom):
    jumlah_kuadrat = np.sum(matriks_keputusan[:, kolom]**2)
    pembagi_topsis[kolom] = np.sqrt(jumlah_kuadrat)
    
    cetak_kuadrat = " + ".join([f"({val}**2)" for val in matriks_keputusan[:, kolom]])
    print(f"Kriteria {nama_kriteria[kolom]}:")
    print(f"  Pembagi = sqrt({cetak_kuadrat}) = {pembagi_topsis[kolom]:.4f}")
    
    for baris in range(len(nama_alternatif)):
        matriks_normalisasi_topsis[baris, kolom] = matriks_keputusan[baris, kolom] / pembagi_topsis[kolom]
        if baris == 0:
            print(f"  Contoh Hitung {nama_alternatif[baris]}: {matriks_keputusan[baris, kolom]} / {pembagi_topsis[kolom]:.4f} = {matriks_normalisasi_topsis[baris, kolom]:.4f}")
    print()

dataframe_normalisasi_topsis = pd.DataFrame(np.round(matriks_normalisasi_topsis, 4), columns=nama_kriteria)
display(dataframe_normalisasi_topsis)

### Matriks Ternormalisasi Terbobot TOPSIS

In [ ]:
print("=== MATRIKS TERNORMALISASI TERBOBOT TOPSIS ===")
matriks_terbobot_topsis = np.zeros_like(matriks_normalisasi_topsis)

for kolom in range(jumlah_kolom):
    print(f"Kriteria {nama_kriteria[kolom]} (Bobot = {bobot_kriteria[kolom]}):")
    for baris in range(len(nama_alternatif)):
        matriks_terbobot_topsis[baris, kolom] = matriks_normalisasi_topsis[baris, kolom] * bobot_kriteria[kolom]
        if baris == 0:
            print(f"  Contoh Hitung {nama_alternatif[baris]}: {matriks_normalisasi_topsis[baris, kolom]:.4f} * {bobot_kriteria[kolom]} = {matriks_terbobot_topsis[baris, kolom]:.4f}")
    print()

dataframe_terbobot_topsis = pd.DataFrame(np.round(matriks_terbobot_topsis, 4), columns=nama_kriteria)
display(dataframe_terbobot_topsis)

### Penentuan Solusi Ideal Positif dan Negatif

In [ ]:
print("=== SOLUSI IDEAL POSITIF (A+) DAN NEGATIF (A-) ===")
solusi_ideal_positif = np.zeros(jumlah_kolom)
solusi_ideal_negatif = np.zeros(jumlah_kolom)

for kolom in range(jumlah_kolom):
    if status_kriteria[kolom] == 'benefit':
        solusi_ideal_positif[kolom] = np.max(matriks_terbobot_topsis[:, kolom])
        solusi_ideal_negatif[kolom] = np.min(matriks_terbobot_topsis[:, kolom])
        print(f"{nama_kriteria[kolom]} [Benefit]: A+ (Max) = {solusi_ideal_positif[kolom]:.4f}, A- (Min) = {solusi_ideal_negatif[kolom]:.4f}")
    else:
        solusi_ideal_positif[kolom] = np.min(matriks_terbobot_topsis[:, kolom])
        solusi_ideal_negatif[kolom] = np.max(matriks_terbobot_topsis[:, kolom])
        print(f"{nama_kriteria[kolom]} [Cost]: A+ (Min) = {solusi_ideal_positif[kolom]:.4f}, A- (Max) = {solusi_ideal_negatif[kolom]:.4f}")

dataframe_solusi_ideal = pd.DataFrame([
    np.round(solusi_ideal_positif, 4),
    np.round(solusi_ideal_negatif, 4)
], columns=nama_kriteria, index=['Ideal Positif', 'Ideal Negatif'])
display(dataframe_solusi_ideal)

### Perhitungan Jarak Euclidean dan Kedekatan Relatif TOPSIS

In [ ]:
print("=== JARAK EUCLIDEAN DAN KEDEKATAN RELATIF ===")
jarak_positif = np.zeros(len(nama_alternatif))
jarak_negatif = np.zeros(len(nama_alternatif))
kedekatan_relatif = np.zeros(len(nama_alternatif))

for baris in range(len(nama_alternatif)):
    jarak_p = np.sqrt(np.sum((matriks_terbobot_topsis[baris] - solusi_ideal_positif)**2))
    jarak_n = np.sqrt(np.sum((matriks_terbobot_topsis[baris] - solusi_ideal_negatif)**2))
    
    jarak_positif[baris] = jarak_p
    jarak_negatif[baris] = jarak_n
    kedekatan_relatif[baris] = jarak_n / (jarak_n + jarak_p)
    
    print(f"{nama_alternatif[baris]}:")
    print(f"  D+ = {jarak_p:.4f}")
    print(f"  D- = {jarak_n:.4f}")
    print(f"  V = {jarak_n:.4f} / ({jarak_n:.4f} + {jarak_p:.4f}) = {kedekatan_relatif[baris]:.4f}\n")

hasil_topsis = pd.DataFrame({
    'Alternatif': nama_alternatif,
    'Nilai TOPSIS': kedekatan_relatif
}).sort_values('Nilai TOPSIS', ascending=False).reset_index(drop=True)
hasil_topsis.index = hasil_topsis.index + 1
display(hasil_topsis)

### METODE 3: Weighted Aggregated Sum Product Assessment (WASPAS)

### Normalisasi Matriks WASPAS

In [ ]:
print("=== NORMALISASI MATRIKS WASPAS ===")
print("Metode WASPAS menggunakan normalisasi linier yang rumusnya identik dengan metode SAW.")
matriks_normalisasi_waspas = matriks_normalisasi_saw.copy()
dataframe_normalisasi_waspas = pd.DataFrame(np.round(matriks_normalisasi_waspas, 4), columns=nama_kriteria)
display(dataframe_normalisasi_waspas)

### Perhitungan Weighted Sum Model (WSM) dan Weighted Product Model (WPM)

In [ ]:
print("=== PERHITUNGAN WSM (Weighted Sum Model) DAN WPM (Weighted Product Model) ===")
nilai_wsm = np.zeros(len(nama_alternatif))
nilai_wpm = np.zeros(len(nama_alternatif))

for baris in range(len(nama_alternatif)):
    detail_wsm = []
    detail_wpm = []
    for kolom in range(jumlah_kolom):
        detail_wsm.append(f"({matriks_normalisasi_waspas[baris, kolom]:.4f} * {bobot_kriteria[kolom]})")
        detail_wpm.append(f"({matriks_normalisasi_waspas[baris, kolom]:.4f}^{bobot_kriteria[kolom]})")
    
    nilai_wsm[baris] = np.sum(matriks_normalisasi_waspas[baris] * bobot_kriteria)
    nilai_wpm[baris] = np.prod(matriks_normalisasi_waspas[baris] ** bobot_kriteria)
    
    print(f"{nama_alternatif[baris]}:")
    print(f"  WSM = {' + '.join(detail_wsm)} = {nilai_wsm[baris]:.4f}")
    print(f"  WPM = {' * '.join(detail_wpm)} = {nilai_wpm[baris]:.4f}\n")

dataframe_wsm_wpm = pd.DataFrame({
    'Alternatif': nama_alternatif,
    'Nilai WSM': np.round(nilai_wsm, 4),
    'Nilai WPM': np.round(nilai_wpm, 4)
})
display(dataframe_wsm_wpm)

### Perhitungan Preferensi WASPAS dan Perangkingan Akhir

In [ ]:
print("=== PERHITUNGAN PREFERENSI WASPAS (LAMBDA = 0.5) ===")
konstanta_lambda = 0.5
nilai_akhir_waspas = np.zeros(len(nama_alternatif))

for baris in range(len(nama_alternatif)):
    nilai_akhir_waspas[baris] = (konstanta_lambda * nilai_wsm[baris]) + ((1 - konstanta_lambda) * nilai_wpm[baris])
    print(f"{nama_alternatif[baris]}:")
    print(f"  Q = ({konstanta_lambda} * {nilai_wsm[baris]:.4f}) + ({(1 - konstanta_lambda)} * {nilai_wpm[baris]:.4f}) = {nilai_akhir_waspas[baris]:.4f}\n")

hasil_waspas = pd.DataFrame({
    'Alternatif': nama_alternatif,
    'Nilai WASPAS': nilai_akhir_waspas
}).sort_values('Nilai WASPAS', ascending=False).reset_index(drop=True)
hasil_waspas.index = hasil_waspas.index + 1
display(hasil_waspas)

### Rekapitulasi Hasil dan Visualisasi

In [ ]:
fig, axis = plt.subplots(1, 3, figsize=(18, 5))
warna_grafik = ['#2c3e50', '#e74c3c', '#27ae60', '#f39c12', '#8e44ad', '#2980b9', '#d35400', '#c0392b', '#1abc9c', '#34495e']
daftar_metode = [('SAW', nilai_akhir_saw), ('TOPSIS', kedekatan_relatif), ('WASPAS', nilai_akhir_waspas)]
for index_ax, (nama_metode, skor_metode) in enumerate(daftar_metode):
    urutan = np.argsort(-skor_metode)
    label_urut = [nama_alternatif[k] for k in urutan]
    skor_urut = skor_metode[urutan]
    warna_urut = [warna_grafik[k] for k in urutan]
    axis[index_ax].barh(label_urut[::-1], skor_urut[::-1], color=warna_urut[::-1], edgecolor='black')
    axis[index_ax].set_title(f'Peringkat Metode {nama_metode}')
    axis[index_ax].set_xlabel('Skor Akhir')
    for idx_bar, v_bar in enumerate(skor_urut[::-1]):
        axis[index_ax].text(v_bar + 0.005, idx_bar, f'{v_bar:.4f}', va='center', fontsize=9)

plt.suptitle('Perbandingan Pemilihan Laptop dengan 3 Metode SPK', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()